In [135]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
import prophet 
from prophet import Prophet

In [136]:
%reload_ext watermark
%watermark -a "Matheus dos Anjos" --iversions

Author: Matheus dos Anjos

prophet    : 1.1.7
matplotlib : 3.9.2
pandas     : 2.2.2
statsmodels: 0.14.2



In [137]:
dados = pd.read_csv('C:/Users/conta/OneDrive/Desktop/Análise e Previsão de Séries Temporais/dataset.csv', encoding='latin1', sep=';')

In [138]:
dados.shape

(51290, 14)

In [139]:
dados.head()

,ï»¿ID_Pedido,Data_Pedido,ID_Cliente,Segmento,Regiao,Pais,Product ID,Categoria,SubCategoria,Total_Vendas,Quantidade,Desconto,Lucro,Prioridade
0,CA-2012-124891,31-07-2012,RH-19495,Consumidor,New York,United States,TEC-AC-10003033,Tecnologia,Accessories,"2309,65",7,0,"762,1845",Critico
1,IN-2013-77878,05-02-2013,JR-16210,Corporativo,New South Wales,Australia,FUR-CH-10003950,Moveis,Chairs,"3709,395",9,"0,1","-288,765",Critico
2,IN-2013-71249,17-10-2013,CR-12730,Consumidor,Queensland,Australia,TEC-PH-10004664,Tecnologia,Phones,"5175,171",9,"0,1","919,971",Medio
3,ES-2013-1579342,28-01-2013,KM-16375,Home Office,Berlin,Germany,TEC-PH-10004583,Tecnologia,Phones,"2892,51",5,"0,1","-96,54",Medio
4,SG-2013-4320,05-11-2013,RH-9495,Consumidor,Dakar,Senegal,TEC-SHA-10000501,Tecnologia,Copiers,"2832,96",8,0,"311,52",Critico


In [140]:
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ï»¿ID_Pedido  51290 non-null  object
 1   Data_Pedido   51290 non-null  object
 2   ID_Cliente    51290 non-null  object
 3   Segmento      51290 non-null  object
 4   Regiao        51290 non-null  object
 5   Pais          51290 non-null  object
 6   Product ID    51290 non-null  object
 7   Categoria     51290 non-null  object
 8   SubCategoria  51290 non-null  object
 9   Total_Vendas  51290 non-null  object
 10  Quantidade    51290 non-null  int64 
 11  Desconto      51290 non-null  object
 12  Lucro         51290 non-null  object
 13  Prioridade    51290 non-null  object
dtypes: int64(1), object(13)
memory usage: 5.5+ MB


In [141]:
dados['Regiao'].unique()

array(['New York', 'New South Wales', 'Queensland', ..., 'Manicaland',
       'Kabarole', 'Matabeleland North'], dtype=object)

In [142]:
dados['SubCategoria'].unique()

array(['Accessories', 'Chairs', 'Phones', 'Copiers', 'Tables', 'Binders',
       'Supplies', 'Appliances', 'Machines', 'Bookcases', 'Storage',
       'Furnishings', 'Art', 'Paper', 'Envelopes', 'Fasteners', 'Labels'],
      dtype=object)

In [143]:
vendas_NY = dados[dados['Regiao'] == 'New York']
vendas_NY = dados[dados['SubCategoria'] == 'Phones']
vendas_NY = dados[['Data_Pedido', 'Quantidade']]
vendas_NY = vendas_NY.set_index('Data_Pedido')

In [145]:
vendas_NY = vendas_NY.sort_index()

In [146]:
vendas_NY.head()

,Quantidade
Data_Pedido,
01-01-2011,3
01-01-2011,5
01-01-2011,4
01-01-2011,2
01-01-2011,2


In [147]:
first_diff = vendas_NY.Quantidade - vendas_NY.Quantidade.shift(1)
first_diff = first_diff.dropna()

In [150]:
first_diff = pd.DataFrame(data = {'ds': first_diff.index, 'y': first_diff.values})

In [152]:
first_diff['ds'].max()

'31-12-2014'

In [162]:
cutoff_date = '31-12-2014'

In [166]:
amostra_treino = first_diff.loc[first_diff.ds < cutoff_date]
amostra_teste = first_diff.loc[first_diff.ds >= cutoff_date]

In [167]:
amostra_treino.head()

,ds,y
0,01-01-2011,2.0
1,01-01-2011,-1.0
2,01-01-2011,-2.0
3,01-01-2011,0.0
4,01-01-2011,1.0


In [168]:
amostra_treino.head()

,ds,y
0,01-01-2011,2.0
1,01-01-2011,-1.0
2,01-01-2011,-2.0
3,01-01-2011,0.0
4,01-01-2011,1.0


In [169]:
modelo = Prophet()

In [172]:
pd.to_datetime(amostra_treino['ds'], format='%d-%m-%Y')

0       2011-01-01
1       2011-01-01
2       2011-01-01
3       2011-01-01
4       2011-01-01
           ...    
51222   2013-12-31
51223   2013-12-31
51224   2013-12-31
51225   2013-12-31
51226   2013-12-31
Name: ds, Length: 51227, dtype: datetime64[ns]

In [174]:
pd.to_datetime(amostra_treino['ds'], dayfirst=True, format='mixed')

0       2011-01-01
1       2011-01-01
2       2011-01-01
3       2011-01-01
4       2011-01-01
           ...    
51222   2013-12-31
51223   2013-12-31
51224   2013-12-31
51225   2013-12-31
51226   2013-12-31
Name: ds, Length: 51227, dtype: datetime64[ns]

In [175]:
modelo.fit(amostra_treino)

ValueError: time data "13-01-2011" doesn't match format "%m-%d-%Y", at position 565. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.